# Model Evaluation & Validation

Choosing the right metric, avoiding leakage in evaluation, calibrating probabilities, and setting decision thresholds from costs.

## ROC-AUC vs PR-AUC — when each matters

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")  # remove this line in Jupyter
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)

# Imbalanced dataset: 1% positive rate
X, y = make_classification(n_samples=10_000, n_features=10, n_informative=6,
                            weights=[0.99, 0.01], random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=42)

model = LogisticRegression(class_weight="balanced", max_iter=500, random_state=42)
model.fit(X_tr, y_tr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_estimator(model, X_te, y_te, ax=ax1)
ax1.set_title("ROC curve — looks great (AUC≈0.97)\nbut positive rate = 1%")

PrecisionRecallDisplay.from_estimator(model, X_te, y_te, ax=ax2)
ax2.axhline(y_te.mean(), ls="--", color="grey", label=f"No-skill baseline ({y_te.mean():.2f})")
ax2.set_title("PR curve — honest picture\nfor imbalanced data")
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()  # in script: plt.savefig(...)
print("Rule: use PR-AUC when positive rate < 10%")

## Calibration — do probabilities mean probabilities?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=5000, n_features=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

gbm = GradientBoostingClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
gbm_cal = CalibratedClassifierCV(GradientBoostingClassifier(n_estimators=100,
    random_state=42), method="isotonic", cv=5).fit(X_tr, y_tr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0,1],[0,1],"k--", label="Perfect")
for model, label in [(gbm, "GBM (raw)"), (gbm_cal, "GBM (calibrated)")]:
    fp, mp = calibration_curve(y_te, model.predict_proba(X_te)[:,1], n_bins=10)
    ax.plot(mp, fp, "o-", label=label)
ax.set(xlabel="Mean predicted prob", ylabel="Fraction positive",
       title="Calibration: does P(y=1|score=0.7) ≈ 0.70?")
ax.legend()
plt.tight_layout()
plt.show()
print("Calibration matters whenever probabilities drive decisions (risk scores, thresholds)")

## Cost-based decision threshold

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
pipe_lr = LogisticRegression(max_iter=1000).fit(StandardScaler().fit_transform(X_tr), y_tr)
proba = pipe_lr.predict_proba(StandardScaler().fit(X_tr).transform(X_te))[:,1]

# Cost of FP (unnecessary biopsy) vs FN (missed cancer)
cost_fp, cost_fn = 500, 10_000   # £

print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<10} {'Expected cost':<15}")
print("-" * 50)
for thresh in np.arange(0.2, 0.8, 0.1):
    pred = (proba >= thresh).astype(int)
    fp = ((pred==1) & (y_te==0)).sum()
    fn = ((pred==0) & (y_te==1)).sum()
    total_cost = fp * cost_fp + fn * cost_fn
    prec = precision_score(y_te, pred, zero_division=0)
    rec  = recall_score(y_te, pred)
    print(f"{thresh:<12.1f} {prec:<12.3f} {rec:<10.3f} £{total_cost:,}")
print("\nOptimal threshold = min expected cost, not 0.5")